# Iceberg Basics — 08: MERGE INTO (Upsert)


MERGE INTO is Iceberg's DML powerhouse — combining insert, update, and delete
in a single atomic operation. Essential for CDC pipelines.

Topics: basic MERGE, WHEN NOT MATCHED BY SOURCE, MoR vs CoW, performance.


In [1]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Spark 4.0.2 | Iceberg catalog ready
26/04/15 06:33:32 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


[Stage 0:>                                                          (0 + 4) / 4]

26/04/15 06:33:35 WARN TaskSetManager: Stage 1 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
Dataset ready: 100,000 rows


## Step 1 — Setup: Orders Table



In [2]:

spark.sql("DROP TABLE IF EXISTS local.icedb.merge_orders")
(df.writeTo("local.icedb.merge_orders")
   .tableProperty("write.delete.mode", "merge-on-read")
   .tableProperty("write.update.mode", "merge-on-read")
   .tableProperty("write.merge.mode",  "merge-on-read")
   .using("iceberg")
   .createOrReplace())
print(f"Orders table: {spark.table('local.icedb.merge_orders').count():,} rows")
spark.table("local.icedb.merge_orders").groupBy("status").count().orderBy("status").show()


26/04/15 06:33:37 WARN TaskSetManager: Stage 4 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


Orders table: 100,000 rows


[Stage 8:>                                                          (0 + 1) / 1]

+---------+-----+
|   status|count|
+---------+-----+
|cancelled|19852|
|confirmed|19603|
|delivered|20181|
|  pending|20172|
|  shipped|20192|
+---------+-----+



## Step 2 — Basic MERGE: Upsert Pattern

Classic CDC: update existing rows, insert new rows.

In [3]:

import random
random.seed(77)

# 300 updates + 100 new orders
updates = spark.createDataFrame(
    [(i, "shipped", datetime.date(2024,6,1)) for i in random.sample(range(1,100001), 300)] +
    [(200001+i, "pending", datetime.date(2024,7,1)) for i in range(100)],
    ["order_id","new_status","change_date"]
)
updates.createOrReplaceTempView("cdc_updates")

before_count = spark.table("local.icedb.merge_orders").count()
print(f"Before MERGE: {before_count:,} rows")

spark.sql("""
    MERGE INTO local.icedb.merge_orders t
    USING cdc_updates s ON t.order_id = s.order_id
    WHEN MATCHED THEN
        UPDATE SET t.status = s.new_status
    WHEN NOT MATCHED THEN
        INSERT (order_id, customer_id, product, category, country,
                quantity, price, revenue, status, order_date)
        VALUES (s.order_id, 0, 'New', 'Unknown', 'US', 1, 0, 0, s.new_status, s.change_date)
""")

after_count = spark.table("local.icedb.merge_orders").count()
print(f"After MERGE : {after_count:,} rows  (+100 new orders)")


Before MERGE: 100,000 rows
26/04/15 06:33:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


After MERGE : 100,100 rows  (+100 new orders)


## Step 3 — WHEN NOT MATCHED BY SOURCE (Delete unmatched)

Three-way MERGE: update, insert, and delete.

In [4]:

# Only keep rows that exist in source — delete the rest
small_source = df.limit(1000).select("order_id","status","order_date") \
                 .withColumnRenamed("status","new_status")
small_source.createOrReplaceTempView("full_refresh_source")

spark.sql("""
    MERGE INTO local.icedb.merge_orders t
    USING full_refresh_source s ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET t.status = s.new_status
    WHEN NOT MATCHED THEN INSERT (order_id,customer_id,product,category,country,
                                   quantity,price,revenue,status,order_date)
                         VALUES (s.order_id,0,'New','Unknown','US',1,0,0,s.new_status,s.order_date)
    WHEN NOT MATCHED BY SOURCE AND t.order_id > 100000 THEN DELETE
""")
print(f"After 3-way MERGE: {spark.table('local.icedb.merge_orders').count():,} rows")


26/04/15 06:33:46 WARN TaskSetManager: Stage 27 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
After 3-way MERGE: 100,000 rows


## Step 4 — MoR vs CoW Write Modes

Merge-on-Read is faster to write; Copy-on-Write is faster to read.

In [5]:

print("Write modes comparison:")
print()
print("Copy-on-Write (CoW) — default:")
print("  Write: rewrites entire Parquet files for affected rows")
print("  Read : plain Parquet scan, no merge")
print("  Best : read-heavy, low write frequency")
print()
print("Merge-on-Read (MoR):")
print("  Write: appends small delete/update files (fast!)")
print("  Read : merges base + delta files at query time")
print("  Best : write-heavy, streaming, frequent small updates")
print()
print("Configure:")
print("  TBLPROPERTIES('write.delete.mode'='merge-on-read')")
print("  TBLPROPERTIES('write.update.mode'='merge-on-read')")
print("  TBLPROPERTIES('write.merge.mode' ='merge-on-read')")
print()
print("After accumulating MoR files, compact with:")
print("  CALL local.system.rewrite_data_files(table=>'db.table')")


Write modes comparison:

Copy-on-Write (CoW) — default:
  Write: rewrites entire Parquet files for affected rows
  Read : plain Parquet scan, no merge
  Best : read-heavy, low write frequency

Merge-on-Read (MoR):
  Write: appends small delete/update files (fast!)
  Read : merges base + delta files at query time
  Best : write-heavy, streaming, frequent small updates

Configure:
  TBLPROPERTIES('write.delete.mode'='merge-on-read')
  TBLPROPERTIES('write.update.mode'='merge-on-read')
  TBLPROPERTIES('write.merge.mode' ='merge-on-read')

After accumulating MoR files, compact with:
  CALL local.system.rewrite_data_files(table=>'db.table')


## Summary



In [6]:

print("""
-- Basic upsert
MERGE INTO local.db.table t
USING source s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.col = s.col
WHEN NOT MATCHED THEN INSERT *

-- With delete
WHEN NOT MATCHED BY SOURCE THEN DELETE

-- Python API
DeltaTable equivalent does not exist in Iceberg Python —
use spark.sql() for MERGE in Iceberg

Performance tips:
  - Filter source before MERGE (reduce matched rows)
  - Use MoR for frequent small updates
  - Run rewrite_data_files after accumulating MoR files
  - MERGE is most efficient when source << target
""")



-- Basic upsert
MERGE INTO local.db.table t
USING source s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.col = s.col
WHEN NOT MATCHED THEN INSERT *

-- With delete
WHEN NOT MATCHED BY SOURCE THEN DELETE

-- Python API
DeltaTable equivalent does not exist in Iceberg Python —
use spark.sql() for MERGE in Iceberg

Performance tips:
  - Filter source before MERGE (reduce matched rows)
  - Use MoR for frequent small updates
  - Run rewrite_data_files after accumulating MoR files
  - MERGE is most efficient when source << target

